# Lab 03: Single-Bet EV — From Odds to Kelly-Sized Bet

This lab walks you through the fundamental math of sports betting: converting odds to probabilities, computing expected value (EV), and sizing bets with the Kelly Criterion. By the end you will:

- Pull latest odds from TimescaleDB (or use synthetic data)
- Convert American odds to decimal odds and implied probability
- Understand the vig (juice) and overround
- Estimate true probability by removing the vig
- Calculate EV for a single bet
- Apply the Kelly Criterion (full and fractional)
- Compute bet sizes from a bankroll
- Build a portfolio view of multiple bets

---

## Prerequisites

- **Labs 01 and 02 completed** — you can connect to TimescaleDB and query odds_ticks
- Understanding of American odds notation (e.g., -110, +150)
- Basic probability knowledge

### Key Formulas Preview

| Formula | Description |
|---|---|
| `implied_prob = 1 / decimal_odds` | Book's implied probability |
| `overround = sum(implied_probs) - 1` | The vig baked into the odds |
| `true_prob = implied_prob / overround` | Vig-adjusted probability (naive method) |
| `EV = true_prob * (decimal_odds - 1) - (1 - true_prob)` | Expected value per $1 staked |
| `kelly = (true_prob * b - (1 - true_prob)) / b` | Kelly fraction (b = decimal_odds - 1) |

---

## Section 1: Setup — Imports and DB Connection

We'll use the sportsquant betting module for odds conversion, Kelly calculations, and edge detection. We also connect to TimescaleDB to pull live odds data.

In [ ]:
# Cell 4: Core imports
import asyncio
import json
from dataclasses import dataclass

import pandas as pd

from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.kelly import (
    KellyCalculator,
    KellyCalculatorConfig,
    EdgeCalculator,
    EdgeCalculatorConfig,
    BankrollManager,
    BankrollManagerConfig,
)
from sportsquant.core.betting.engine import (
    american_to_decimal,
    calculate_ev,
    detect_arbitrage,
    expected_value,
    kelly_fraction,
)
from sportsquant.util.time_utils import american_to_implied_prob, safe_float
from sportsquant.infra.db.connection import DBConfig, DatabasePool, get_pool, reset_pool

import nest_asyncio
nest_asyncio.apply()

print('Imports loaded successfully.')

In [ ]:
# Cell 5: Connect to TimescaleDB
#
# If the DB is not available, we'll fall back to synthetic data below.

db_available = False
try:
    config = DBConfig.from_env()
    pool = await get_pool(config)
    db_available = True
    print(f'Connected to {config.host}:{config.port}/{config.database}')
except Exception as e:
    print(f'DB not available: {e}')
    print('Will use synthetic data for this lab.')

---

## Section 2: Pull Latest Odds from the Database

We'll query the `odds_ticks` table for the most recent head-to-head (h2h) odds. If the database is empty or unavailable, we'll use synthetic data.

In [ ]:
# Cell 7: Fetch latest h2h odds from odds_ticks
#
# We look for head-to-head (moneyline) odds. Each row has:
#   sport, league, event_id, book, market, selection, price, line, ts

if db_available:
    rows = await pool.fetch(
        "SELECT sport, league, event_id, book, market, selection, price, line, ts "
        "FROM odds_ticks WHERE market = 'h2h' ORDER BY ts DESC LIMIT 20"
    )
    if rows:
        odds_df = pd.DataFrame([dict(r) for r in rows])
        print(f'Loaded {len(odds_df)} h2h odds rows from DB')
    else:
        print('DB has no h2h odds data. Using synthetic data.')
        db_available = False

if not db_available or not len(rows):
    # Synthetic data representing typical NFL moneyline odds
    synthetic_data = [
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'KC_vs_BAL_2026', 'book': 'draftkings', 'market': 'h2h', 'selection': 'KC Chiefs', 'price': -150, 'line': None, 'ts': '2026-06-17T18:00:00Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'KC_vs_BAL_2026', 'book': 'draftkings', 'market': 'h2h', 'selection': 'BAL Ravens', 'price': +130, 'line': None, 'ts': '2026-06-17T18:00:00Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'KC_vs_BAL_2026', 'book': 'fanduel', 'market': 'h2h', 'selection': 'KC Chiefs', 'price': -145, 'line': None, 'ts': '2026-06-17T18:00:05Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'KC_vs_BAL_2026', 'book': 'fanduel', 'market': 'h2h', 'selection': 'BAL Ravens', 'price': +125, 'line': None, 'ts': '2026-06-17T18:00:05Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'SF_vs_DET_2026', 'book': 'draftkings', 'market': 'h2h', 'selection': 'SF 49ers', 'price': -110, 'line': None, 'ts': '2026-06-17T18:01:00Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'SF_vs_DET_2026', 'book': 'draftkings', 'market': 'h2h', 'selection': 'DET Lions', 'price': +105, 'line': None, 'ts': '2026-06-17T18:01:00Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'BUF_vs_MIA_2026', 'book': 'betmgm', 'market': 'h2h', 'selection': 'BUF Bills', 'price': -200, 'line': None, 'ts': '2026-06-17T18:02:00Z'},
        {'sport': 'nfl', 'league': 'NFL', 'event_id': 'BUF_vs_MIA_2026', 'book': 'betmgm', 'market': 'h2h', 'selection': 'MIA Dolphins', 'price': +175, 'line': None, 'ts': '2026-06-17T18:02:00Z'},
    ]
    odds_df = pd.DataFrame(synthetic_data)
    print(f'Using {len(odds_df)} synthetic odds rows')

odds_df.head(10)

---

## Section 3: Parse American Odds

American odds come in two flavors:

- **Negative** (e.g., -110): you must bet $110 to win $100. Implies the favorite.
- **Positive** (e.g., +150): you win $150 for every $100 bet. Implies the underdog.

The conversion to **decimal odds** (used in EV calculations):

```
Positive American:  decimal = 1 + (american / 100)
Negative American:  decimal = 1 + (100 / |american|)
```

And to **implied probability**:

```
Positive:  implied_prob = 100 / (american + 100)
Negative:  implied_prob = |american| / (|american| + 100)
```

In [ ]:
# Cell 9: Convert American odds to decimal and implied probability
#
# We use sportsquant's Odds class for the conversion.

# Let's pick a few example odds and convert them
example_odds = [-150, +130, -110, +105, -200, +175]

print(f"{'American':<12} {'Decimal':<12} {'Implied Prob':<15} {'Implied %':<12}")
print(f"{'-'*12} {'-'*12} {'-'*15} {'-'*12}")

for amer in example_odds:
    odds_obj = Odds(american=amer)
    dec = odds_obj.to_decimal()
    imp_prob = odds_obj.implied_prob()
    print(f"{amer:<12} {dec:<12.4f} {imp_prob:<15.6f} {imp_prob * 100:<12.2f}%")

# Also show using the utility function
print('\nUsing american_to_implied_prob from time_utils:')
for amer in example_odds:
    prob = american_to_implied_prob(amer)
    print(f'  american={amer:+4d} → implied_prob={prob:.6f} ({prob*100:.2f}%)')

---

## Section 4: The Vig (Juice)

Bookmakers charge a commission called the **vig** (or **juice**). It's built into the odds: the implied probabilities of both sides sum to more than 1.0.

For a typical -110 / -110 market:
- Implied prob of each side: 110 / (110 + 100) = 0.5238
- Sum: 0.5238 + 0.5238 = 1.0476
- Overround: 0.0476 (4.76%)

The **overround** is the total excess over 1.0. It represents the book's edge.

In [ ]:
# Cell 11: Calculate overround (vig) for each event
#
# Group by event and book, then sum implied probabilities for h2h markets.

def calculate_overround(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate overround for each event+book combination.

    Args:
        df: DataFrame with columns 'event_id', 'book', 'price'.

    Returns:
        DataFrame with overround per event+book.
    """
    results = []
    for (event_id, book), group in df.groupby(['event_id', 'book']):
        implied_probs = []
        for _, row in group.iterrows():
            prob = american_to_implied_prob(row['price'])
            if prob is not None:
                implied_probs.append(prob)
        if implied_probs:
            overround = sum(implied_probs) - 1.0
            results.append({
                'event_id': event_id,
                'book': book,
                'implied_sum': sum(implied_probs),
                'overround': overround,
                'overround_pct': f'{overround * 100:.2f}%',
            })
    return pd.DataFrame(results)

vig_df = calculate_overround(odds_df)
print('Overround (vig) by event and book:')
print(vig_df.to_string(index=False))
print(f"\nAverage overround: {vig_df['overround'].mean():.4f} ({vig_df['overround'].mean()*100:.2f}%)")

---

## Section 5: Estimate True Probability

The simplest way to remove the vig is the **naive method**: divide each implied probability by the overround (sum of all implied probabilities).

```
true_prob = implied_prob / overround_sum
```

This ensures the probabilities sum to exactly 1.0. More sophisticated methods (Shin's method, power method) exist but the naive method is a good starting point.

The `EdgeCalculator` in `sportsquant.core.betting.kelly` provides edge and EV calculations, but for true probability estimation we'll implement the naive method here.

In [ ]:
# Cell 13: Estimate true probability using the naive method
#
# For each event+book, we normalize implied probabilities by the overround.

def estimate_true_probabilities(df: pd.DataFrame) -> pd.DataFrame:
    """Estimate true probabilities by removing the vig (naive method).

    Args:
        df: DataFrame with columns 'event_id', 'book', 'selection', 'price'.

    Returns:
        DataFrame with added 'decimal_odds', 'implied_prob', 'true_prob' columns.
    """
    result_rows = []
    for (event_id, book), group in df.groupby(['event_id', 'book']):
        implied_probs = []
        decimal_odds_list = []
        for _, row in group.iterrows():
            odds_obj = Odds(american=int(row['price']))
            dec = odds_obj.to_decimal()
            imp = odds_obj.implied_prob()
            decimal_odds_list.append(dec)
            implied_probs.append(imp)
        
        # Naive method: divide by overround
        overround_sum = sum(implied_probs)
        true_probs = [p / overround_sum for p in implied_probs]
        
        for (_, row), dec, imp, true in zip(group.iterrows(), decimal_odds_list, implied_probs, true_probs):
            result_rows.append({
                'event_id': row['event_id'],
                'book': row['book'],
                'selection': row['selection'],
                'price': row['price'],
                'decimal_odds': dec,
                'implied_prob': imp,
                'true_prob': true,
            })
    
    return pd.DataFrame(result_rows)

prob_df = estimate_true_probabilities(odds_df)
print('True probability estimates (naive vig removal):')
print(prob_df.to_string(index=False))
print(f"\nVerification — true probs sum per event+book:")
for (event_id, book), group in prob_df.groupby(['event_id', 'book']):
    print(f'  {event_id} @ {book}: sum = {group["true_prob"].sum():.6f}')

---

## Section 6: Calculate Expected Value

The **expected value (EV)** of a bet tells you how much you expect to profit per dollar wagered over the long run:

```
EV = true_prob × (decimal_odds - 1) - (1 - true_prob)
```

- **EV > 0**: You have an edge (positive expected profit)
- **EV < 0**: The book has an edge (negative expected profit)
- **EV = 0**: Break-even bet

We use `sportsquant.core.betting.engine.expected_value()` and `EdgeCalculator.compute_expected_value()` to compute EV.

In [ ]:
# Cell 15: Calculate EV for each selection
#
# We use both the engine's expected_value() and EdgeCalculator.

edge_calc = EdgeCalculator()

def compute_ev_row(row: dict) -> dict:
    """Compute EV and edge for a single bet."""
    odds_obj = Odds(american=int(row['price']))
    true_prob = row['true_prob']
    decimal_odds = row['decimal_odds']
    
    # Method 1: engine.expected_value
    ev_engine = expected_value(true_prob, odds_obj, true_prob)
    
    # Method 2: EdgeCalculator.compute_expected_value
    ev_edge = edge_calc.compute_expected_value(true_prob, decimal_odds, stake=1.0)
    
    # Method 3: EdgeCalculator.compute_edge
    edge = edge_calc.compute_edge(true_prob, decimal_odds)
    
    return {
        **row,
        'ev': ev_engine,
        'ev_alt': ev_edge,
        'edge': edge,
        'edge_pct': f'{edge * 100:.2f}%',
    }

ev_df = prob_df.apply(lambda row: pd.Series(compute_ev_row(row.to_dict())), axis=1)
print('EV calculations for each selection:')
print(ev_df[['selection', 'book', 'price', 'decimal_odds', 'true_prob', 'ev', 'edge_pct']].to_string(index=False))

# Highlight +EV bets
positive_ev = ev_df[ev_df['ev'] > 0]
if not positive_ev.empty:
    print(f'\n🎯 Found {len(positive_ev)} +EV bet(s):')
    for _, row in positive_ev.iterrows():
        print(f'  {row["selection"]} @ {row["book"]} (odds={row["price"]}, EV={row["ev"]:+.4f}, edge={row["edge"]*100:.2f}%)')
else:
    print('\nNo +EV bets found in the current data.')

---

## Section 7: The Kelly Criterion

The **Kelly Criterion** determines the optimal fraction of your bankroll to bet to maximize long-term growth:

```
kelly_fraction = (true_prob × b - (1 - true_prob)) / b
```

where `b = decimal_odds - 1` (the net odds).

- **kelly > 0**: You have an edge → bet this fraction of your bankroll
- **kelly ≤ 0**: No edge → don't bet

**Fractional Kelly** (e.g., ¼ Kelly) is commonly used to reduce variance. We'll use `KellyCalculator` from `sportsquant.core.betting.kelly`.

In [ ]:
# Cell 17: Calculate Kelly fractions for +EV bets
#
# We use KellyCalculator from sportsquant.core.betting.kelly.

kelly_calc = KellyCalculator()

def compute_kelly_row(row: dict) -> dict:
    """Compute Kelly fractions for a single bet."""
    true_prob = row['true_prob']
    decimal_odds = row['decimal_odds']
    
    # Full Kelly
    full_kelly = kelly_calc.compute_kelly(true_prob, decimal_odds)
    
    # Quarter Kelly (0.25x)
    quarter_kelly = kelly_calc.compute_fractional_kelly(true_prob, decimal_odds, fraction=0.25)
    
    # Half Kelly (0.5x)
    half_kelly = kelly_calc.compute_fractional_kelly(true_prob, decimal_odds, fraction=0.5)
    
    return {
        **row,
        'full_kelly': full_kelly,
        'half_kelly': half_kelly,
        'quarter_kelly': quarter_kelly,
    }

kelly_df = ev_df.apply(lambda row: pd.Series(compute_kelly_row(row.to_dict())), axis=1)

print('Kelly fractions for each selection:')
print(kelly_df[['selection', 'book', 'price', 'true_prob', 'ev', 'full_kelly', 'half_kelly', 'quarter_kelly']].to_string(index=False))

# Show only +EV bets with their Kelly fractions
positive_kelly = kelly_df[kelly_df['full_kelly'] > 0]
if not positive_kelly.empty:
    print(f'\n✅ {len(positive_kelly)} selection(s) with positive Kelly:')
    for _, row in positive_kelly.iterrows():
        print(f'  {row["selection"]} @ {row["book"]}: full_kelly={row["full_kelly"]:.4f}, '
              f'half={row["half_kelly"]:.4f}, quarter={row["quarter_kelly"]:.4f}')

---

## Section 8: Bet Sizing

Given a bankroll and Kelly fraction, the actual bet size is:

```
bet_size = bankroll × kelly_fraction
```

The `BankrollManager` in `sportsquant.core.betting.kelly` also applies position limits and exposure constraints.

In [ ]:
# Cell 19: Compute bet sizes with BankrollManager
#
# We use a $10,000 bankroll and apply position limits.

BANKROLL = 10_000.0

bm = BankrollManager(bankroll=BANKROLL)

def compute_bet_size(row: dict) -> dict:
    """Compute recommended bet size for a single selection."""
    true_prob = row['true_prob']
    decimal_odds = row['decimal_odds']
    full_kelly = row['full_kelly']
    
    # Use quarter Kelly for actual sizing (more conservative)
    quarter_kelly = row['quarter_kelly']
    
    if quarter_kelly > 0:
        bet_size = bm.compute_bet_size(
            kelly_fraction=quarter_kelly,
            odds=decimal_odds,
            win_probability=true_prob,
        )
    else:
        bet_size = 0.0
    
    return {
        **row,
        'bankroll': BANKROLL,
        'quarter_kelly_bet': round(bet_size, 2),
        'expected_profit': round(bet_size * row['ev'], 2),
    }

sized_df = kelly_df.apply(lambda row: pd.Series(compute_bet_size(row.to_dict())), axis=1)

print(f'Bankroll: ${BANKROLL:,.0f}')
print(f"Max position: {bm.config.exposure_limits.max_position_pct * 100:.0f}%")
print()
print('Bet sizing (quarter-Kelly):')
print(sized_df[['selection', 'book', 'price', 'ev', 'quarter_kelly', 'quarter_kelly_bet', 'expected_profit']].to_string(index=False))

---

## Section 9: The Edge Concept

**Edge** is the difference between your estimated true probability and the book's implied probability:

```
edge = true_prob - implied_prob
```

If `true_prob > implied_prob`, you have a positive edge — the book has undervalued this outcome. The `EdgeCalculator` computes this as:

```
edge = (true_prob × decimal_odds) - 1
```

which is equivalent to `true_prob - implied_prob` (since `implied_prob = 1 / decimal_odds`).

In [ ]:
# Cell 21: Detailed edge analysis
#
# We compare true_prob vs implied_prob for each selection.

print('Edge analysis: true_prob vs. implied_prob')
print(f"{'Selection':<15} {'Book':<15} {'Price':<8} {'Impl Prob':<12} {'True Prob':<12} {'Edge':<10} {'Verdict':<10}")
print(f"{'-'*15} {'-'*15} {'-'*8} {'-'*12} {'-'*12} {'-'*10} {'-'*10}")

for _, row in sized_df.iterrows():
    edge_val = row['edge']
    verdict = '+EDGE' if edge_val > 0 else 'NO EDGE'
    print(f"{row['selection']:<15} {row['book']:<15} {row['price']:<8} "
          f"{row['implied_prob']:<12.6f} {row['true_prob']:<12.6f} {edge_val:<10.6f} {verdict:<10}")

---

## Section 10: Putting It All Together

Let's wrap the entire pipeline into a single function that takes (odds, true_prob, bankroll) and returns (EV, kelly_frac, bet_size).

In [ ]:
# Cell 23: Complete EV + Kelly + sizing pipeline

@dataclass
class BetAnalysis:
    """Complete analysis of a single bet opportunity."""
    selection: str
    american_odds: int
    decimal_odds: float
    implied_prob: float
    true_prob: float
    ev: float
    edge: float
    full_kelly: float
    quarter_kelly: float
    bet_size: float
    expected_profit: float


def analyze_bet(
    selection: str,
    american_odds: int,
    true_prob: float,
    bankroll: float = 10_000.0,
    kelly_fraction: float = 0.25,
) -> BetAnalysis:
    """Analyze a single bet from odds to sizing.

    Args:
        selection: Name of the selection (team/player).
        american_odds: American odds (e.g., -110, +150).
        true_prob: Your estimated true probability (0-1).
        bankroll: Current bankroll in dollars.
        kelly_fraction: Fraction of Kelly to use (default 0.25 = quarter-Kelly).

    Returns:
        BetAnalysis with all computed fields.
    """
    odds_obj = Odds(american=american_odds)
    decimal_odds = odds_obj.to_decimal()
    implied_prob = odds_obj.implied_prob()
    
    # EV and edge
    ev_val = expected_value(true_prob, odds_obj, true_prob)
    edge_val = edge_calc.compute_edge(true_prob, decimal_odds)
    
    # Kelly fractions
    full_k = kelly_calc.compute_kelly(true_prob, decimal_odds)
    frac_k = kelly_calc.compute_fractional_kelly(true_prob, decimal_odds, fraction=kelly_fraction)
    
    # Bet sizing
    bm = BankrollManager(bankroll=bankroll)
    bet_size = bm.compute_bet_size(
        kelly_fraction=frac_k,
        odds=decimal_odds,
        win_probability=true_prob,
    )
    
    return BetAnalysis(
        selection=selection,
        american_odds=american_odds,
        decimal_odds=decimal_odds,
        implied_prob=implied_prob,
        true_prob=true_prob,
        ev=ev_val,
        edge=edge_val,
        full_kelly=full_k,
        quarter_kelly=frac_k,
        bet_size=round(bet_size, 2),
        expected_profit=round(bet_size * ev_val, 2),
    )


# Quick test
result = analyze_bet('KC Chiefs', -150, true_prob=0.65, bankroll=10_000)
print(f'Analysis for {result.selection}:')
print(f'  American odds: {result.american_odds}')
print(f'  Decimal odds:  {result.decimal_odds:.4f}')
print(f'  Implied prob:  {result.implied_prob:.4f}')
print(f'  True prob:     {result.true_prob:.4f}')
print(f'  EV:            {result.ev:+.4f}')
print(f'  Edge:          {result.edge:+.4f} ({result.edge*100:.2f}%)')
print(f'  Full Kelly:    {result.full_kelly:.4f}')
print(f'  Quarter Kelly: {result.quarter_kelly:.4f}')
print(f'  Bet size:      ${result.bet_size:.2f}')
print(f'  Expected profit: ${result.expected_profit:.2f}')

---

## Section 11: Multiple Bets — Portfolio View

Now let's apply the full pipeline to several rows from our odds data and build a portfolio view of all potential bets.

In [ ]:
# Cell 25: Build a portfolio of analyzed bets
#
# For each unique event+book combination, we analyze both sides.
# In practice, you'd only bet the side with +EV.

portfolio = []
for _, row in prob_df.iterrows():
    analysis = analyze_bet(
        selection=row['selection'],
        american_odds=int(row['price']),
        true_prob=row['true_prob'],
        bankroll=10_000,
        kelly_fraction=0.25,
    )
    portfolio.append(analysis)

# Build a summary DataFrame
portfolio_data = [{
    'selection': a.selection,
    'odds': a.american_odds,
    'dec_odds': f'{a.decimal_odds:.3f}',
    'true_prob': f'{a.true_prob:.4f}',
    'EV': f'{a.ev:+.4f}',
    'edge': f'{a.edge:+.4f}',
    'q_kelly': f'{a.quarter_kelly:.4f}',
    'bet_size': f'${a.bet_size:.2f}',
    'exp_profit': f'${a.expected_profit:.2f}',
} for a in portfolio]

portfolio_df = pd.DataFrame(portfolio_data)
print('Portfolio Summary (quarter-Kelly sizing, $10K bankroll):')
print(portfolio_df.to_string(index=False))

# Total expected profit
total_expected = sum(a.expected_profit for a in portfolio if a.ev > 0)
positive_bets = sum(1 for a in portfolio if a.ev > 0)
print(f'\nPositive EV bets: {positive_bets}/{len(portfolio)}')
print(f'Total expected profit from +EV bets: ${total_expected:.2f}')

---

## Exercises

Try these on your own:

1. **Find a +EV bet in current data** — Modify the `true_prob` values used in the portfolio analysis. Try increasing a team's probability by 5% and see how the EV and Kelly change. At what point does the bet become +EV?

2. **Compare fractional Kelly strategies** — The lab uses quarter-Kelly (0.25x). Re-run the portfolio analysis with half-Kelly (0.5x) and full Kelly (1.0x). How do the bet sizes and expected profits change? Which strategy would you prefer and why?

3. **Sensitivity analysis** — Write a function that plots EV as a function of true_prob for a given set of American odds. For odds of -110, what true probability do you need to have a +EV bet?

4. **Arbitrage detection** — Use `detect_arbitrage()` from `sportsquant.core.betting.engine` to check if any of the event+book combinations offer an arbitrage opportunity. What does an arbitrage opportunity look like in terms of implied probabilities?

5. **Multi-bet Kelly** — Use `KellyCalculator.compute_kelly_multi_bet()` to calculate Kelly fractions for multiple bets at once. How does it handle correlated bets?

---

## Summary

In this lab you learned:

- How to convert American odds to decimal odds and implied probabilities
- How to calculate and interpret the vig (overround)
- How to estimate true probability by removing the vig
- How to calculate Expected Value (EV) for a single bet
- How the Kelly Criterion determines optimal bet sizing
- How to apply fractional Kelly (quarter-Kelly) for risk management
- How to use `BankrollManager` for position-limited sizing
- How to build a portfolio view of multiple bets

### Key API Reference

| Class/Function | Module | Purpose |
|---|---|---|
| `Odds` | `sportsquant.core.betting.odds` | Convert American/decimal odds |
| `KellyCalculator` | `sportsquant.core.betting.kelly` | Full, fractional, adaptive Kelly |
| `EdgeCalculator` | `sportsquant.core.betting.kelly` | Compute edge and EV |
| `BankrollManager` | `sportsquant.core.betting.kelly` | Position-limited bet sizing |
| `expected_value()` | `sportsquant.core.betting.engine` | EV calculation |
| `kelly_fraction()` | `sportsquant.core.betting.engine` | Quick Kelly fraction |
| `american_to_decimal()` | `sportsquant.core.betting.engine` | Odds conversion |
| `detect_arbitrage()` | `sportsquant.core.betting.engine` | Find arb opportunities |
| `american_to_implied_prob()` | `sportsquant.util.time_utils` | Odds → probability |

### Next Steps

Continue to **Lab 04: Multi-Book Middling** to learn how to detect and size middle opportunities across multiple sportsbooks.

---

*Don't forget to close the pool:*
```python
await pool.close()
```

In [ ]:
# Cell 28: Close the connection pool
if db_available:
    await pool.close()
    print('Connection pool closed. Lab 03 complete!')
else:
    print('Lab 03 complete! (used synthetic data, no DB connection to close)')